In [0]:
# Databricks notebook source
# DBTITLE 1,Import the Required Library
import requests 
import json 
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
# DBTITLE 1,Create Catalog , Schema and Volume
spark.sql("CREATE CATALOG IF NOT EXISTS crickdata")
spark.sql("CREATE SCHEMA IF NOT EXISTS crickdata.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS crickdata.bronze.cricket_api_project")

base_path='/Volumes/crickdata/bronze/cricket_api_project'

In [0]:
API_KEY='f53d866b-c240-44d4-ae2b-a3e451ffc9ed'
api_url=f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"

response =requests.get(api_url)
response.raise_for_status()

api_data=response.json()
print(api_data.keys())
print(json.dumps(api_data,indent= 2)[:2000])

In [0]:
raw_file_path=f'{base_path}/current_matches_raw.json'

with open (raw_file_path,'w') as file:
    json.dump(api_data,file)

print("RAW API data is save at the :",raw_file_path)

In [0]:

bronze_schema=StructType([
StructField("source_api",StringType(),True),
StructField("raw_json",StringType(),True),
StructField("ingestion_time",TimestampType(),True)
])
bronze_data=[{
"source_api":api_url,
"raw_json":json.dumps(api_data),
"ingestion_time":None
}]


In [0]:
bronze_df=spark.createDataFrame(bronze_data,bronze_schema)\
    .withColumn("ingestion_time",current_timestamp())

display(bronze_df)

In [0]:
bronze_df.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable("crickdata.bronze.cricket_bronze_current_matches")

print("BRONZE TABLE CREATED SUCCESSFULLY")

In [0]:
%sql
select * from crickdata.bronze.cricket_bronze_current_matches